##### 1. PACKAGE SETUP

In [8]:
# Import libraries for:
# - data handling (pandas, numpy)
# - text preprocessing (nltk)
# - linguistic annotation (spaCy)
# - vectorization and similarity (scikit-learn)
# - deep semantic embeddings (sentence-transformers)

import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords, opinion_lexicon
from nltk.tokenize import word_tokenize

import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

In [21]:
import sys
!{sys.executable} -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 2.9 MB/s eta 0:00:05
     ---- ----------------------------------- 1.3/12.8 MB 4.5 MB/s eta 0:00:03
     ---- ----------------------------------- 1.6/12.8 MB 4.1 MB/s eta 0:00:03
     ------ --------------------------------- 2.1/12.8 MB 2.9 MB/s eta 0:00:04
     --------- ------------------------------ 3.1/12.8 MB 3.2 MB/s eta 0:00:03
     ------------- -------------------------- 4.2/12.8 MB 3.6 MB/s eta 0:00:03
     ------------------ --------------------- 5.8/12.8 MB 4.2 MB/s eta 0:00:02
     ---------------------- ----------------- 7.1/12.8 MB 4.5 MB/s eta 0:00:02
     --------------------------- ------------ 8.7/12.8 MB 4.9 MB/s eta 0:00:01
     ------------------------------- -------- 10.0/12.8 MB 5.1 MB/s eta 0:00:01
     ------------------------------------ --- 11.5/12.8 MB 

In [13]:
# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab') 
nltk.download('stopwords')
nltk.download('opinion_lexicon')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Dhrumil\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Dhrumil\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dhrumil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package opinion_lexicon to
[nltk_data]     C:\Users\Dhrumil\AppData\Roaming\nltk_data...
[nltk_data]   Package opinion_lexicon is already up-to-date!


True

##### 2. INPUT TEXTS

In [10]:
# Create a small dataset of 4 documents to demonstrate the NLP pipeline
text_df = pd.DataFrame({
    "doc_id": [1, 2, 3, 4],
    "text": [
        "Apple opened a new office in Toronto. Employees said the workplace feels exciting and modern.",
        "The airline delayed my flight for six hours. The staff were polite, but the experience was frustrating.",
        "Dr. Sarah Chen from the University of British Columbia presented a climate report in Vancouver.",
        "I thought the movie would be amazing. It was visually beautiful, but the story felt empty."
    ]
})

In [11]:
text_df

,doc_id,text
0,1,Apple opened a new office in Toronto. Employee...
1,2,The airline delayed my flight for six hours. T...
2,3,Dr. Sarah Chen from the University of British ...
3,4,I thought the movie would be amazing. It was v...


##### 3. LEXICAL LAYER (basic word-level analysis)

In [14]:
# Tokenize each document into individual words
text_df["tokens"] = text_df["text"].apply(word_tokenize)

In [15]:
# Remove common stopwords (e.g., "the", "is") and keep only meaningful words
stop_words = set(stopwords.words('english'))

tokens_words = (
    text_df[["doc_id", "tokens"]]
    .explode("tokens")                  # convert list of tokens into rows
    .rename(columns={"tokens": "word"})
)

In [16]:
# Normalize to lowercase and filter out stopwords and non-alphabetic tokens
tokens_words["word"] = tokens_words["word"].str.lower()
tokens_words = tokens_words[~tokens_words["word"].isin(stop_words)]
tokens_words = tokens_words[tokens_words["word"].str.isalpha()]

In [17]:
# Count word frequencies across all documents
word_freq = (
    tokens_words["word"]
    .value_counts()
    .reset_index()
)
word_freq.columns = ["word", "n"]

In [18]:
print("\nTop words")
print(word_freq.head(20))


Top words
           word  n
0         apple  1
1     vancouver  1
2          chen  1
3    university  1
4       british  1
5      columbia  1
6     presented  1
7       climate  1
8        report  1
9       thought  1
10       opened  1
11        movie  1
12        would  1
13      amazing  1
14     visually  1
15    beautiful  1
16        story  1
17         felt  1
18        sarah  1
19  frustrating  1


##### 4. LEXICAL DIVERSITY

In [19]:
# Measure vocabulary richness using:
# - total_words: total number of tokens
# - unique_words: number of distinct words
# - type_token_ratio (TTR): diversity metric = unique / total

def lexical_stats_fn(tokens):
    total_words = len(tokens)
    unique_words = len(set(tokens))
    ttr = unique_words / total_words if total_words > 0 else 0
    return pd.Series([total_words, unique_words, ttr])

lexical_stats = text_df.copy()
lexical_stats[["total_words", "unique_words", "type_token_ratio"]] = \
    lexical_stats["tokens"].apply(lexical_stats_fn)

lexical_stats = lexical_stats[["doc_id", "total_words", "unique_words", "type_token_ratio"]]

print("\nLexical diversity")
print(lexical_stats)


Lexical diversity
   doc_id  total_words  unique_words  type_token_ratio
0       1         17.0          16.0          0.941176
1       2         20.0          18.0          0.900000
2       3         16.0          16.0          1.000000
3       4         19.0          17.0          0.894737


##### 5. SYNTAX LAYER (POS tagging using spaCy)

In [22]:
# Analyze grammatical structure by assigning:
# - lemma (base form of word)
# - POS tag (noun, verb, etc.)

nlp = spacy.load("en_core_web_sm")

rows = []
for _, row in text_df.iterrows():
    doc = nlp(row["text"])
    for token in doc:
        rows.append({
            "doc_id": row["doc_id"],
            "token": token.text,
            "lemma": token.lemma_,
            "upos": token.pos_   # universal POS tag
        })

anno_df = pd.DataFrame(rows)

print("\nPOS tags preview")
print(anno_df[["doc_id", "token", "lemma", "upos"]].head(30))


POS tags preview
    doc_id      token      lemma   upos
0        1      Apple      Apple  PROPN
1        1     opened       open   VERB
2        1          a          a    DET
3        1        new        new    ADJ
4        1     office     office   NOUN
5        1         in         in    ADP
6        1    Toronto    Toronto  PROPN
7        1          .          .  PUNCT
8        1  Employees   employee   NOUN
9        1       said        say   VERB
10       1        the        the    DET
11       1  workplace  workplace   NOUN
12       1      feels       feel   VERB
13       1   exciting   exciting    ADJ
14       1        and        and  CCONJ
15       1     modern     modern    ADJ
16       1          .          .  PUNCT
17       2        The        the    DET
18       2    airline    airline   NOUN
19       2    delayed      delay   VERB
20       2         my         my   PRON
21       2     flight     flight   NOUN
22       2        for        for    ADP
23       2        six 

##### 6. SENTIMENT LAYER

In [23]:
# Use a lexicon-based approach:
# - match words against predefined positive/negative lists
# - compute sentiment counts per document
pos_words = set(opinion_lexicon.positive())
neg_words = set(opinion_lexicon.negative())

def get_sentiment(word):
    if word in pos_words:
        return "positive"
    elif word in neg_words:
        return "negative"
    return None

tokens_words["sentiment"] = tokens_words["word"].apply(get_sentiment)

# Keep only words that carry sentiment
sentiment_df = tokens_words.dropna(subset=["sentiment"])

# Aggregate sentiment counts per document
sentiment_df = (
    sentiment_df
    .groupby(["doc_id", "sentiment"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Compute overall sentiment score (positive - negative)
sentiment_df["net_sentiment"] = sentiment_df.get("positive", 0) - sentiment_df.get("negative", 0)

print("\nSentiment output")
print(sentiment_df)


Sentiment output
sentiment  doc_id  negative  positive  net_sentiment
0               1         0         2              2
1               2         2         1             -1
2               4         0         2              2


##### 7. NER PROXY (Named Entity approximation)

In [24]:
# Instead of full NER, extract proper nouns (PROPN) as a simple proxy
# This captures names of people, places, organizations, etc.
ner_proxy = (
    anno_df[anno_df["upos"] == "PROPN"]
    .groupby("doc_id")["token"]
    .apply(lambda x: ", ".join(sorted(set(x))))
    .reset_index()
)

print("\nNamed-entity proxy")
print(ner_proxy)


Named-entity proxy
   doc_id                                              token
0       1                                     Apple, Toronto
1       3  British, Chen, Columbia, Dr., Sarah, Universit...


##### 8. TF-IDF SEMANTIC SIMILARITY

In [25]:
# Convert documents into TF-IDF vectors:
# - captures importance of words relative to corpus
# Then compute cosine similarity between documents
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(text_df["text"])

sim_mat_tfidf = cosine_similarity(tfidf_matrix)

print("\nTF-IDF similarity")
print(np.round(sim_mat_tfidf, 4))


TF-IDF similarity
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


##### 9. SENTENCE-TRANSFORMER EMBEDDINGS

In [26]:
# Use pretrained transformer model to capture deeper semantic meaning
# Each document is converted into a dense vector embedding
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(text_df["text"].tolist())

# Compute cosine similarity using embeddings (captures context better than TF-IDF)
sim_mat_st = cosine_similarity(embeddings)

print("\nSentence-transformer similarity")
print(np.round(sim_mat_st, 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Dhrumil\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dhrumil\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Sentence-transformer similarity
[[ 1.      0.189   0.0444  0.0986]
 [ 0.189   1.      0.0102  0.0907]
 [ 0.0444  0.0102  1.     -0.0242]
 [ 0.0986  0.0907 -0.0242  1.    ]]


##### 10. COMBINED SUMMARY TABLE

In [27]:
# Merge outputs from different layers into a single table:
# - lexical statistics
# - sentiment scores
# - named entity proxy
summary_df = lexical_stats.merge(sentiment_df, on="doc_id", how="left") \
                          .merge(ner_proxy, on="doc_id", how="left")

print("\nCombined summary")
print(summary_df)


Combined summary
   doc_id  total_words  unique_words  type_token_ratio  negative  positive  \
0       1         17.0          16.0          0.941176       0.0       2.0   
1       2         20.0          18.0          0.900000       2.0       1.0   
2       3         16.0          16.0          1.000000       NaN       NaN   
3       4         19.0          17.0          0.894737       0.0       2.0   

   net_sentiment                                              token  
0            2.0                                     Apple, Toronto  
1           -1.0                                                NaN  
2            NaN  British, Chen, Columbia, Dr., Sarah, Universit...  
3            2.0                                                NaN  
